# Análise de Negócio — IMDb Movies Dataset
## Perguntas 1 a 9 | Python e Pandas

**Objetivo:** investigar fatores associados à nota do público, diferenças entre aceitação e faturamento e limitações dos dados antes de recomendar investimentos.

### Como usar
1. Abra o arquivo no **Google Colab** ou Jupyter Notebook.
2. Coloque `imdb_movies.csv` na mesma pasta do notebook ou faça upload quando solicitado.
3. Execute as células em ordem (`Ambiente de execução → Executar tudo` no Colab).
4. Leia as conclusões automáticas como **evidências exploratórias**, não como prova de causalidade.

**Atenção:** este dataset não contém colunas estruturadas para diretor, produtor, estúdio, marketing, número de salas ou franquia. A coluna `crew` aparenta misturar nomes de atores e personagens; portanto, ela não é suficiente para medir o desempenho de diretores/equipes. Os resultados financeiros também serão submetidos a verificações de qualidade.

In [ ]:
import os
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)
pd.set_option("display.max_colwidth", 80)

# Procura o CSV em caminhos comuns. Se não encontrar, abre seletor de upload no Colab.
possible_paths = [
    "imdb_movies.csv",
    "/content/imdb_movies.csv",
    "/mnt/data/imdb_movies.csv"
]
csv_path = next((p for p in possible_paths if os.path.exists(p)), None)

if csv_path is None:
    try:
        from google.colab import files
        uploaded = files.upload()
        csv_path = next((name for name in uploaded if name.lower().endswith(".csv")), None)
    except Exception:
        csv_path = input("Digite o caminho completo do arquivo imdb_movies.csv: ").strip()

if not csv_path:
    raise FileNotFoundError("Não foi possível localizar imdb_movies.csv. Faça upload do arquivo e execute novamente.")

df_raw = pd.read_csv(csv_path)
df = df_raw.copy()

print(f"Arquivo carregado: {csv_path}")
print(f"Linhas: {df.shape[0]:,} | Colunas: {df.shape[1]}")
display(df.head())

## Etapa 0 — Conhecer e preparar os dados

Antes de responder às perguntas, verificamos os tipos, valores ausentes, duplicatas e a consistência das variáveis principais. Os nomes das colunas são mantidos como vieram no arquivo.

In [ ]:
print("Colunas:")
print(df.columns.tolist())

print("\nTipos e valores não nulos:")
display(pd.DataFrame({
    "tipo": df.dtypes.astype(str),
    "não_nulos": df.notna().sum(),
    "nulos": df.isna().sum(),
    "%_nulos": (df.isna().mean() * 100).round(2)
}).sort_values("%_nulos", ascending=False))

print("\nLinhas duplicadas completas:", df.duplicated().sum())

# Conversões numéricas seguras
for col in ["score", "budget_x", "revenue"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# Data: o formato pode variar; errors='coerce' marca datas não interpretadas como NaT.
if "date_x" in df.columns:
    df["date_parsed"] = pd.to_datetime(df["date_x"], errors="coerce")
    # Tenta também o formato mês/dia/ano quando a primeira tentativa não reconhece a data.
    mask = df["date_parsed"].isna()
    df.loc[mask, "date_parsed"] = pd.to_datetime(df.loc[mask, "date_x"], errors="coerce", dayfirst=False)

print("\nResumo das variáveis numéricas:")
display(df.select_dtypes(include=np.number).describe().T)

### Verificação de qualidade dos dados financeiros

Valores zero, negativos, extremos e combinações orçamento/receita repetidas podem ser legítimos em alguns casos, mas também podem indicar dados ausentes codificados como zero, duplicação ou problemas na fonte. Vamos medir isso antes de interpretar os resultados.

In [ ]:
financial_cols = [c for c in ["budget_x", "revenue"] if c in df.columns]
quality_rows = []
for col in financial_cols:
    s = df[col]
    quality_rows.append({
        "variável": col,
        "nulos": int(s.isna().sum()),
        "zeros": int((s == 0).sum()),
        "negativos": int((s < 0).sum()),
        "positivos": int((s > 0).sum()),
        "mínimo": s.min(),
        "mediana": s.median(),
        "máximo": s.max()
    })
display(pd.DataFrame(quality_rows))

if set(["budget_x", "revenue"]).issubset(df.columns):
    pair_counts = df.groupby(["budget_x", "revenue"], dropna=False).size().sort_values(ascending=False)
    repeated_pairs = pair_counts[pair_counts > 1]
    repeated_pair_rows = int(pair_counts[pair_counts > 1].sum()) if len(repeated_pairs) else 0
    print(f"Combinações orçamento/receita que se repetem: {len(repeated_pairs):,}")
    print(f"Registros pertencentes a pares repetidos: {repeated_pair_rows:,}")
    if len(repeated_pairs):
        print("Exemplos de pares repetidos:")
        display(repeated_pairs.head(10).rename("quantidade de registros").reset_index())

if "score" in df.columns:
    print("\nNotas fora do intervalo 0–100:", int(((df["score"] < 0) | (df["score"] > 100)).sum()))
    print("Filmes com nota exatamente 0:", int((df["score"] == 0).sum()))
    print("Filmes com nota exatamente 100:", int((df["score"] == 100).sum()))

### Funções auxiliares

As funções abaixo serão reutilizadas nas perguntas para evitar repetir código. Para correlações, usamos somente pares de valores válidos. Correlação mede associação linear, não causalidade.

In [ ]:
def corr_pair(data, x, y):
    temp = data[[x, y]].replace([np.inf, -np.inf], np.nan).dropna()
    return temp[x].corr(temp[y]) if len(temp) >= 2 else np.nan

def format_num(value, digits=2):
    return "não disponível" if pd.isna(value) else f"{value:,.{digits}f}"

def plot_scatter_with_trend(data, x, y, title, xlabel, ylabel, log_x=False):
    temp = data[[x, y]].replace([np.inf, -np.inf], np.nan).dropna()
    if log_x:
        temp = temp[(temp[x] > 0)]
        x_values = np.log10(temp[x])
        x_label = f"log10({xlabel})"
    else:
        x_values = temp[x]
        x_label = xlabel
    plt.figure(figsize=(9, 5.5))
    plt.scatter(x_values, temp[y], alpha=0.25, s=14)
    if len(temp) >= 2 and x_values.nunique() > 1:
        coef = np.polyfit(x_values, temp[y], 1)
        xx = np.linspace(x_values.min(), x_values.max(), 100)
        plt.plot(xx, np.polyval(coef, xx), linewidth=2)
    plt.title(title)
    plt.xlabel(x_label)
    plt.ylabel(ylabel)
    plt.grid(alpha=0.2)
    plt.tight_layout()
    plt.show()

def genre_explode(data):
    if "genre" not in data.columns:
        return pd.DataFrame(columns=["genre_item", "score", "budget_x", "revenue"])
    cols = [c for c in ["score", "budget_x", "revenue"] if c in data.columns]
    temp = data[["genre"] + cols].copy()
    temp["genre_item"] = temp["genre"].fillna("Não informado").astype(str).str.replace("\u00a0", " ", regex=False).str.split(r",\s*")
    temp = temp.explode("genre_item")
    temp["genre_item"] = temp["genre_item"].astype(str).str.strip()
    return temp

# Pergunta 1
## Além do gênero, o que mais parece influenciar a nota que um filme recebe?

Vamos comparar nota com orçamento, receita, ano e idioma/país quando disponíveis. Primeiro medimos correlações numéricas; depois observamos diferenças entre gêneros. A palavra **influenciar** será tratada com cuidado: este dataset observacional permite identificar associações, não provar causas.

In [ ]:
numeric_candidates = [c for c in ["score", "budget_x", "revenue"] if c in df.columns]
if "date_parsed" in df.columns:
    df["year"] = df["date_parsed"].dt.year
    numeric_candidates.append("year")

print("Correlação das variáveis numéricas com a nota:")
corr_rows = []
for col in numeric_candidates:
    if col != "score":
        corr_rows.append({"variável": col, "correlação com score": corr_pair(df, col, "score")})
corr_table = pd.DataFrame(corr_rows).sort_values("correlação com score", key=lambda s: s.abs(), ascending=False)
display(corr_table.round(3))

if "budget_x" in df.columns:
    plot_scatter_with_trend(df, "budget_x", "score",
        "Orçamento x nota IMDb", "Orçamento", "Nota IMDb", log_x=True)
if "revenue" in df.columns:
    plot_scatter_with_trend(df, "revenue", "score",
        "Receita x nota IMDb", "Receita", "Nota IMDb", log_x=True)

genres = genre_explode(df)
if "score" in genres.columns:
    genre_stats = (genres.dropna(subset=["score"])
                   .groupby("genre_item")["score"]
                   .agg(media="mean", mediana="median", quantidade="count")
                   .query("quantidade >= 30")
                   .sort_values("media"))
    display(genre_stats.round(2))
    plt.figure(figsize=(9, 7))
    plt.barh(genre_stats.index, genre_stats["media"])
    plt.xlabel("Nota média")
    plt.ylabel("Gênero")
    plt.title("Nota média por gênero (mínimo de 30 registros)")
    plt.grid(axis="x", alpha=0.2)
    plt.tight_layout()
    plt.show()

print("\nConclusão sugerida:")
if "budget_x" in df.columns:
    r_budget = corr_pair(df, "budget_x", "score")
    print(f"- Correlação orçamento/nota: {r_budget:.3f}. Isso não sustenta a ideia de que orçamento maior garanta nota maior.")
if "revenue" in df.columns:
    r_revenue = corr_pair(df, "revenue", "score")
    print(f"- Correlação receita/nota: {r_revenue:.3f}. Receita e avaliação parecem medir dimensões diferentes.")
print("- As médias por gênero ajudam a identificar diferenças, mas há variação dentro de cada gênero; gênero não determina sozinho a nota.")

# Pergunta 2
## Existe algo que esperávamos encontrar e não encontramos? O que faríamos se precisássemos disso?

O objetivo é identificar lacunas do dataset. A lista abaixo compara variáveis disponíveis com variáveis úteis para uma decisão de investimento.

In [ ]:
available = set(df.columns)
expected_fields = {
    "Diretor": ["director", "directors"],
    "Produtor": ["producer", "producers"],
    "Roteirista": ["writer", "writers", "screenwriter"],
    "Estúdio": ["studio", "production_company", "production_companies"],
    "Franquia/série": ["franchise", "series", "collection"],
    "Marketing/publicidade": ["marketing_budget", "marketing_spend", "advertising"],
    "Número de salas": ["theaters", "number_of_theaters", "screens"],
    "Bilheteria de estreia": ["opening_weekend", "opening_gross"],
    "Nota dos críticos": ["critic_score", "metascore", "rotten_tomatoes"],
    "Número de votos": ["vote_count", "num_votes", "votes"]
}
gap_table = []
for concept, candidates in expected_fields.items():
    found = [c for c in candidates if c in available]
    gap_table.append({"informação desejada": concept,
                      "coluna encontrada": ", ".join(found) if found else "Não encontrada"})
display(pd.DataFrame(gap_table))

print("Colunas presentes no dataset:", ", ".join(df.columns))
print("\nConclusão sugerida:")
print("O arquivo tem título, nota, gênero, sinopse, país/idioma, orçamento e receita, mas não traz de forma estruturada vários fatores que explicariam melhor o sucesso.")
print("Para preencher as lacunas, integraríamos fontes adicionais e manteríamos uma chave confiável por filme (por exemplo, ID externo), validando títulos e anos.")

# Pergunta 3
## Filmes muito bem avaliados costumam ter algo em comum além de gênero e orçamento?

Definiremos “muito bem avaliado” como nota igual ou superior a 80. Esse corte é uma escolha analítica, não uma regra universal. Compararemos grupo de alta nota com os demais em orçamento, idioma, país, período e gêneros.

In [ ]:
valid_score = df.dropna(subset=["score"]).copy()
high = valid_score[valid_score["score"] >= 80]
other = valid_score[valid_score["score"] < 80]
print(f"Filmes com nota >= 80: {len(high):,} ({len(high)/len(valid_score)*100:.1f}% dos filmes com nota válida)")
print(f"Filmes com nota < 80: {len(other):,}")

comparison = []
for label, subset in [("Nota >= 80", high), ("Nota < 80", other)]:
    row = {"grupo": label, "quantidade": len(subset), "nota_mediana": subset["score"].median()}
    for col in ["budget_x", "revenue", "year"]:
        if col in subset.columns:
            row[f"{col}_mediana"] = subset[col].median()
    for col in ["orig_lang", "country"]:
        if col in subset.columns:
            row[f"{col}_mais_comum"] = subset[col].mode(dropna=True).iloc[0] if subset[col].notna().any() else np.nan
    comparison.append(row)
display(pd.DataFrame(comparison).round(2))

if "genre" in df.columns:
    g = genre_explode(df[["genre", "score"]].copy())
    high_genres = g[g["score"] >= 80].groupby("genre_item").size().rename("filmes nota >= 80")
    all_genres = g.groupby("genre_item").size().rename("total filmes")
    genre_high = pd.concat([all_genres, high_genres], axis=1).fillna(0)
    genre_high["% com nota >= 80"] = genre_high["filmes nota >= 80"] / genre_high["total filmes"] * 100
    genre_high = genre_high[genre_high["total filmes"] >= 30].sort_values("% com nota >= 80", ascending=False)
    display(genre_high.head(15).round(2))

print("\nConclusão sugerida:")
if len(high) and len(other) and "budget_x" in df.columns:
    print(f"- Mediana do orçamento dos filmes com nota >= 80: {high['budget_x'].median():,.0f}; dos demais: {other['budget_x'].median():,.0f}.")
print("- Compare também idioma, país, ano e distribuição por gênero; diferenças são pistas para investigação, não causas comprovadas.")
print("- Para uma conclusão mais sólida, controlaríamos orçamento, gênero, ano e popularidade, e verificaríamos se os resultados se mantêm.")

# Pergunta 4
## A informação sobre quem trabalhou em cada filme é suficiente para explicar destaques consistentes?

Vamos inspecionar a coluna `crew` e verificar se existem campos estruturados para direção e produção. A resposta deve considerar o conteúdo real, não apenas o nome da coluna.

In [ ]:
print("Campos relacionados a equipe/créditos:", [c for c in df.columns if any(k in c.lower() for k in ["crew", "director", "producer", "writer", "cast", "actor"])])
if "crew" in df.columns:
    display(df[["names", "crew"]].head(8))
    print("\nExemplo de conteúdo da coluna crew:")
    print(str(df["crew"].dropna().iloc[0])[:500] if df["crew"].notna().any() else "Coluna sem valores preenchidos")

structured_roles = [c for c in ["director", "producer", "writer", "production_company", "studio"] if c in df.columns]
print("\nCampos estruturados de direção/produção/estúdio encontrados:", structured_roles if structured_roles else "Nenhum")
print("\nConclusão sugerida:")
print("Não é suficiente. A coluna crew aparenta combinar nomes de pessoas e personagens, sem separar claramente atores, direção, roteiro e produção.")
print("Não devemos atribuir o sucesso a um diretor ou produtor usando essa coluna sem uma etapa de enriquecimento e validação.")

# Pergunta 5
## Como descobrir um padrão de sucesso ligado às pessoas por trás dos filmes?

Esta pergunta é principalmente metodológica. O dataset atual não permite uma resposta confiável sobre diretores/produtores, mas podemos desenhar a análise que faríamos após complementar os dados.

In [ ]:
plan = pd.DataFrame([
    ["1. Enriquecer créditos", "Adicionar diretor, roteirista, produtor e estúdio a partir de uma fonte estruturada."],
    ["2. Identificar filmes", "Usar ID único e validar título + ano para evitar juntar remakes ou títulos iguais."],
    ["3. Definir métricas", "Calcular quantidade de filmes, mediana da nota, proporção com nota >= 80 e receita mediana."],
    ["4. Controlar contexto", "Comparar profissionais dentro de gêneros, faixas de orçamento, anos e franquias semelhantes."],
    ["5. Validar estabilidade", "Separar treino/teste temporal ou testar em filmes posteriores para evitar conclusões baseadas em poucos sucessos."],
    ["6. Evitar viés", "Exigir número mínimo de filmes e mostrar incerteza; não ranquear alguém com base em um único título."]
], columns=["etapa", "ação"])
display(plan)

print("Conclusão sugerida:")
print("Com o dataset atual, não é possível identificar um padrão confiável de diretores ou produtores.")
print("Com créditos estruturados, compararíamos desempenho consistente e controlaríamos gênero, orçamento, período e franquia.")

# Pergunta 6
## Os filmes que mais faturaram são também os mais bem avaliados? Sempre?

Vamos listar os filmes de maior receita e medir a proporção de títulos de alta receita que têm nota abaixo de 70. A receita é usada como faturamento bruto informado pelo dataset; não é lucro.

In [ ]:
valid_finance = df.dropna(subset=["revenue", "score"]).copy()
valid_finance = valid_finance[valid_finance["revenue"] > 0]
top_revenue = valid_finance.sort_values("revenue", ascending=False).head(15)
display_cols = [c for c in ["names", "date_x", "score", "budget_x", "revenue", "country"] if c in top_revenue.columns]
display(top_revenue[display_cols].style.format({
    "score": "{:.1f}", "budget_x": "${:,.0f}", "revenue": "${:,.0f}"
}))

if len(valid_finance):
    cutoff = valid_finance["revenue"].quantile(0.90)
    top_decile = valid_finance[valid_finance["revenue"] >= cutoff]
    below70 = (top_decile["score"] < 70).mean() * 100
    print(f"Limite do 10% de maior receita: {cutoff:,.0f}")
    print(f"Entre os 10% de maior receita, {below70:.1f}% têm nota abaixo de 70.")
    plot_scatter_with_trend(valid_finance, "revenue", "score",
                            "Receita x nota: faturamento não é reputação",
                            "Receita", "Nota IMDb", log_x=True)

print("\nConclusão sugerida:")
print("Não. Há filmes de grande faturamento com notas altas e outros com notas medianas. Receita bruta não equivale a lucro nem a satisfação do público.")

# Pergunta 7
## O que explicaria um filme ou sequência que fatura muito, mas recebe nota mediana ou baixa?

Vamos identificar filmes que combinam receita alta com nota abaixo de 70. As explicações (franquia, marketing, distribuição, personagens conhecidos) são hipóteses de negócio: o dataset não contém todas essas variáveis para testá-las diretamente.

In [ ]:
valid_finance = df.dropna(subset=["revenue", "score"]).copy()
valid_finance = valid_finance[valid_finance["revenue"] > 0]
if len(valid_finance):
    revenue_cut = valid_finance["revenue"].quantile(0.90)
    high_revenue_middling = valid_finance[
        (valid_finance["revenue"] >= revenue_cut) & (valid_finance["score"] < 70)
    ].sort_values("revenue", ascending=False)
    display(high_revenue_middling[[c for c in ["names", "date_x", "score", "budget_x", "revenue", "genre"] if c in df.columns]].head(20))
    print(f"Casos identificados: {len(high_revenue_middling)}")

print("\nHipóteses a investigar com dados adicionais:")
for item in [
    "Franquia ou personagens conhecidos geram demanda prévia.",
    "Marketing e distribuição ampla podem aumentar o público alcançado.",
    "A sequência pode atrair fãs do filme anterior mesmo com recepção mais fraca.",
    "A receita depende também de preço, mercados e duração em cartaz; a nota mede avaliação de quem votou.",
    "Sem dados de marketing, salas, bilheteria de estreia e franquia, não conseguimos confirmar qual hipótese explica cada caso."
]:
    print("-", item)

print("\nConclusão sugerida:")
print("O faturamento alto com nota mediana é plausível porque receita e avaliação medem coisas diferentes. Para explicar cada caso, precisamos cruzar a lista com dados de franquia, marketing, distribuição e bilheteria por mercado.")

# Pergunta 8
## Filmes de uma mesma série/continuação se comportam de forma parecida em nota e receita?

O dataset não possui uma coluna de franquia. O trecho abaixo usa um **mapa manual editável** de palavras-chave para encontrar possíveis títulos relacionados. Isso é uma aproximação exploratória: revise os resultados antes de apresentar e não trate a classificação automática como verdade.

In [ ]:
# Edite, acrescente ou remova padrões conforme a lista de filmes do seu dataset.
# Os padrões são procurados no título, ignorando maiúsculas/minúsculas.
franchise_patterns = {
    "Avatar": r"\bavatar\b",
    "Avengers": r"\bavengers\b",
    "Star Wars": r"\bstar wars\b",
    "Jurassic": r"\bjurassic\b",
    "Harry Potter": r"\bharry potter\b",
    "Fast & Furious": r"\b(fast.*furious|fast & furious)\b",
    "Mission: Impossible": r"\bmission: impossible\b",
    "Transformers": r"\btransformers\b",
    "Spider-Man": r"\bspider.?man\b",
    "Batman": r"\bbatman\b",
    "John Wick": r"\bjohn wick\b",
    "Halloween": r"\bhalloween\b",
    "Scream": r"\bscream\b",
    "Toy Story": r"\btoy story\b",
    "Shrek": r"\bshrek\b",
    "Frozen": r"\bfrozen\b",
    "The Hunger Games": r"\bhunger games\b",
    "Twilight": r"\btwilight\b",
    "Pirates of the Caribbean": r"\bpirates of the caribbean\b",
    "The Lord of the Rings": r"\blord of the rings\b",
    "The Hobbit": r"\bthe hobbit\b",
    "Terminator": r"\bterminator\b",
    "Planet of the Apes": r"\bplanet of the apes\b",
    "Creed/Rocky": r"\b(creed|rocky)\b"
}

title_col = "names" if "names" in df.columns else "orig_title"
fr = df.copy()
fr["franchise_candidate"] = pd.NA
for franchise, pattern in franchise_patterns.items():
    mask = fr[title_col].fillna("").astype(str).str.contains(pattern, case=False, regex=True)
    # Não sobrescreve uma classificação já atribuída.
    fr.loc[mask & fr["franchise_candidate"].isna(), "franchise_candidate"] = franchise

candidate_counts = fr["franchise_candidate"].value_counts(dropna=True)
print("Franquias candidatas encontradas:")
display(candidate_counts.rename("quantidade de filmes").to_frame())

franchise_data = fr[fr["franchise_candidate"].notna()].copy()
if len(franchise_data):
    franchise_summary = franchise_data.groupby("franchise_candidate").agg(
        quantidade=("score", "count"),
        nota_media=("score", "mean"),
        nota_min=("score", "min"),
        nota_max=("score", "max"),
        receita_mediana=("revenue", "median"),
        receita_min=("revenue", "min"),
        receita_max=("revenue", "max")
    ).sort_values("quantidade", ascending=False)
    display(franchise_summary.round(2))

    # Visualiza somente franquias candidatas com pelo menos 2 filmes e dados suficientes.
    repeated = franchise_summary[franchise_summary["quantidade"] >= 2].sort_values("nota_media")
    if len(repeated):
        plt.figure(figsize=(9, 5.5))
        plt.barh(repeated.index, repeated["nota_media"])
        plt.xlabel("Nota média")
        plt.title("Nota média por franquia candidata (classificação por título)")
        plt.grid(axis="x", alpha=0.2)
        plt.tight_layout()
        plt.show()

print("\nConclusão sugerida:")
print("A classificação por palavras-chave ajuda a criar uma primeira lista, mas pode incluir falsos positivos e deixar sequências de fora.")
print("Para uma conclusão confiável, construiríamos uma tabela validada de filme → franquia e compararíamos evolução de nota e receita ao longo dos lançamentos.")

# Pergunta 9
## Quanto maior o orçamento, melhor a nota? Qual é a relação e por que pode ser assim?

Vamos calcular correlação, comparar faixas de orçamento e visualizar a distribuição. Valores zero ou ausentes não devem ser interpretados automaticamente como orçamento real; por isso, também apresentamos uma análise apenas para orçamentos positivos.

In [ ]:
budget_score = df.dropna(subset=["budget_x", "score"]).copy()
budget_score = budget_score[budget_score["budget_x"] > 0]

if len(budget_score) >= 2:
    r = budget_score["budget_x"].corr(budget_score["score"])
    print(f"Correlação orçamento/nota, apenas orçamento positivo: {r:.3f}")

    # Faixas por quartis; duplicates='drop' evita erro quando os cortes coincidem.
    budget_score["faixa_orcamento"] = pd.qcut(
        budget_score["budget_x"], q=4, duplicates="drop"
    )
    budget_groups = budget_score.groupby("faixa_orcamento", observed=True)["score"].agg(
        quantidade="count", nota_media="mean", nota_mediana="median"
    )
    display(budget_groups.round(2))

    plt.figure(figsize=(9, 5.5))
    plt.scatter(np.log10(budget_score["budget_x"]), budget_score["score"], alpha=0.25, s=14)
    plt.xlabel("log10(Orçamento positivo)")
    plt.ylabel("Nota IMDb")
    plt.title("Orçamento x nota IMDb (somente orçamento positivo)")
    plt.grid(alpha=0.2)
    plt.tight_layout()
    plt.show()

    if abs(r) < 0.1:
        interpretation = "A correlação linear é muito fraca."
    elif abs(r) < 0.3:
        interpretation = "A correlação linear é fraca."
    elif abs(r) < 0.5:
        interpretation = "A correlação linear é moderada."
    else:
        interpretation = "A correlação linear é relativamente forte; ainda assim, não prova causalidade."
    print("\nInterpretação:", interpretation)
    print("O orçamento pode financiar elenco, efeitos, produção e escala, mas não garante roteiro, direção ou conexão emocional com o público.")
else:
    print("Não há dados suficientes com orçamento positivo e nota válida para calcular a relação.")

print("\nConclusão sugerida:")
print("Os dados não sustentam a ideia simples de que gastar mais garante nota maior. Antes de decidir investimento, seria melhor combinar orçamento com gênero, equipe criativa, franquia, marketing, distribuição e desempenho histórico.")

# Síntese executiva
## O que recomendar ao cliente?

A célula final consolida as principais métricas calculadas neste notebook. Revise os resultados e as limitações antes de transformar a análise em recomendação de investimento.

In [ ]:
summary_lines = []
if "budget_x" in df.columns and "score" in df.columns:
    rb = corr_pair(df[df["budget_x"] > 0], "budget_x", "score")
    summary_lines.append(f"- Correlação entre orçamento positivo e nota: **{rb:.3f}**.")
if "revenue" in df.columns and "score" in df.columns:
    rr = corr_pair(df[df["revenue"] > 0], "revenue", "score")
    summary_lines.append(f"- Correlação entre receita positiva e nota: **{rr:.3f}**.")
if "score" in df.columns:
    summary_lines.append(f"- Registros com nota válida: **{df['score'].notna().sum():,}**.")
if "budget_x" in df.columns and "score" in df.columns:
    hi = df[df["score"] >= 80]
    lo = df[df["score"] < 80]
    if len(hi) and len(lo):
        summary_lines.append(f"- Mediana do orçamento para nota >= 80: **{hi['budget_x'].median():,.0f}**; para nota < 80: **{lo['budget_x'].median():,.0f}**.")
summary_lines += [
    "- Nota alta e faturamento alto são objetivos diferentes e devem ser medidos separadamente.",
    "- As relações observadas são exploratórias; correlação não prova causalidade.",
    "- O dataset precisa ser complementado com créditos de direção/produção, franquia, estúdio, marketing e distribuição.",
    "- Valores financeiros repetidos, zeros e extremos precisam ser validados antes de apoiar decisões reais."
]
display(Markdown("### Principais conclusões\n" + "\n".join(summary_lines)))

print("\nChecklist antes da apresentação:")
for item in [
    "Conferir se os gráficos e tabelas foram executados sem erro.",
    "Validar manualmente os filmes citados como exemplos.",
    "Explicar que receita bruta não é lucro.",
    "Mencionar limitações e inconsistências dos dados.",
    "Separar o que os dados mostram das hipóteses que ainda precisam ser testadas."
]:
    print("☐", item)